# DPPUv2 paper03: Paper Figures
## Unified Geometric Landau EFT of Homogeneous S³×S¹ Minisuperspace in Einstein-Cartan + Nieh-Yan Theory

This notebook generates all numerically computed figures for paper03.

### Figure map
| Figure | Section | Content | Cell(s) |
|--------|---------|---------|----------|
| Fig. 4 | §3 | MX sector Pontryagin density $P_0(r)$ | Fig4 |
| Fig. 5 | §4 | Spin-2 quintet $T_A$ basis (heatmaps) | Fig5 |
| Fig. 7 | §4 | Spin-2 mass splitting vs $\varepsilon$ (closed-form) | CFscan, Fig7 |
| Fig. 8 | §4 | $T_1$ soft mode at $\varepsilon_{c+}=\sqrt{11/5}-1$ (exact) | CFscan, Fig8 |
| Fig. 9 | §4 | $s\neq0$ phase transition (negative-$\varepsilon$ side) | Engine, Fig9 |
| Fig. 11 | §5 | $G$ field-space metric eigenvalue spectrum | VelEngine, Fig11 |
| Fig. 12 | §5 | $(\omega,\delta)\to(X,Y)$ basis rotation schematic | Fig12 |

**Author:** Muacca  
**Date:** 2026-03-17 (Figs 7, 8 updated to closed-form analytical)

---
> **Note on computation time:**
> - `CFscan` cells: closed-form analytical scan (~5 s first run, then instant from cache).  
> - Cells marked `[SLOW]` run a SymPy symbolic engine (~2–4 min each) on first run.  
> - The velocity-mode engine cell `[VERY SLOW]` takes ~300–600 s on first run.  
> - All results are cached in `output/paper03/figures_cache.pkl`.  
>   Cache keys: `spin2_cf_scan` (Figs 7, 8), `spin2_scan` (Fig. 9), `G_metric` (Fig. 11).

In [ ]:
import sys
import os
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as mpatches
from scipy.optimize import minimize_scalar, brentq
from scipy.linalg import eigh

# ── Matplotlib style ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 110,
    'savefig.dpi': 200,
})

# ── Path setup ─────────────────────────────────────────────────
# Locate the bundled script root whether Jupyter starts in the paper root
# or in script/scripts/visualize/.
from pathlib import Path

def find_project_root():
    cwd = Path.cwd().resolve()
    bases = [cwd, *cwd.parents]
    candidates = bases + [base / 'script' for base in bases]
    for candidate in candidates:
        if (candidate / 'dppu').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise RuntimeError('Could not locate paper03 script root containing dppu/ and scripts/.')

notebook_dir = str(Path.cwd().resolve())
project_root = str(find_project_root())
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"Project root: {project_root}")
latex_fig_dir = os.path.normpath(os.path.join(project_root, '..', 'LaTeX', 'figures'))
os.makedirs(latex_fig_dir, exist_ok=True)

cache_file  = os.path.join(project_root, 'figures_cache.pkl')

# ── Cache helpers ──────────────────────────────────────────────
def load_cache():
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    return {}

def save_cache(c):
    with open(cache_file, 'wb') as f:
        pickle.dump(c, f)

cache = load_cache()
print(f"Cached keys : {list(cache.keys())}")
print(f"Output dir  : {latex_fig_dir}")

In [ ]:
# ── Spin-2 quintet T_A (traceless symmetric 3×3, orthonormal) ──
sqrt2 = np.sqrt(2)
sqrt6 = np.sqrt(6)

T = {}
T[1] = np.array([[1, 0, 0], [0, 1, 0], [0, 0, -2]], float) / sqrt6   # m=0
T[2] = np.array([[1, 0, 0], [0, -1, 0], [0, 0, 0]], float) / sqrt2   # m=+2
T[3] = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], float) / sqrt2    # m=-2
T[4] = np.array([[0, 0, 1], [0, 0, 0], [1, 0, 0]], float) / sqrt2    # m=+1
T[5] = np.array([[0, 0, 0], [0, 0, 1], [0, 1, 0]], float) / sqrt2    # m=-1

T_labels = {
    1: r'$T_1$' + '\n' + r'$(m=0)$',
    2: r'$T_2$' + '\n' + r'$(m=+2)$',
    3: r'$T_3$' + '\n' + r'$(m=-2)$',
    4: r'$T_4$' + '\n' + r'$(m=+1)$',
    5: r'$T_5$' + '\n' + r'$(m=-1)$',
}

ok = all(
    abs(np.trace(T[A] @ T[B]) - (1 if A == B else 0)) < 1e-12
    for A in range(1, 6) for B in range(1, 6)
)
print(f"T_A orthonormality Tr(T_A T_B) = delta_AB : {'OK' if ok else 'FAIL'}")

## Fig. 4 — §3: Pontryagin density in the MX sector

**Source:** analytical formula  
**Section:** §3 (Proposition MX-P)

$$P_0 = \frac{2V\eta\,(V^2r^2 + 9\eta^2 - 36)}{9r^3}$$

This shows that $P_0 = 0$ iff $V=0$ (AX mode), $\eta=0$ (VT mode), or the
specific zeros of the numerator.  Only MX mode ($V\neq0$, $\eta\neq0$) gives
$P_0\neq0$; the origin is the torsion mixing product $V\eta$.

In [ ]:
# ── Fig. 4 — Pontryagin density P₀(r) in MX sector ──────────
plt.close('all')
r_arr = np.linspace(0.3, 7.0, 500)

def P0(r, V, eta):
    """Isotropic-point Pontryagin density P₀ (App B.4)."""
    return 2 * V * eta * (V**2 * r**2 + 9 * eta**2 - 36) / (9 * r**3)

# (V, eta, label, color, linestyle)
param_sets = [
    (1,  1,  r'$V=1,\,\eta=1$',    'tab:blue',   '-'),
    (2,  1,  r'$V=2,\,\eta=1$',    'tab:orange',  '--'),
    (1,  2,  r'$V=1,\,\eta=2$',    'tab:green',  '-.'),
    (1, -1,  r'$V=1,\,\eta=-1$',   'tab:red',    ':'),
]

fig, ax = plt.subplots(figsize=(7, 4.5))
for V, eta, label, color, ls in param_sets:
    ax.plot(r_arr, P0(r_arr, V, eta), color=color, ls=ls, lw=2, label=label)

ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel(r'$r$')
ax.set_ylabel(r'$P_0$')
ax.set_ylim(-3.5, 3.5)
ax.set_xlim(r_arr[0], r_arr[-1])
ax.legend(loc='upper right')

# Mark P=0 reference
ax.text(6.8, 0.12, r'$P=0$' + '\n(AX/VT)', fontsize=10, ha='right', color='gray')

ax.set_title(
    r'Fig. 4 — Pontryagin density $P_0(r)$ in MX sector'
    '\n' + r'($P\equiv 0$ when $V=0$ or $\eta=0$)',
    fontsize=12
)

plt.tight_layout()
plt.savefig(os.path.join(latex_fig_dir, 'fig04_pontryagin_mx.png'), bbox_inches='tight')
plt.show()
print("Fig. 4 saved.")

## Fig. 5 — §4: Spin-2 quintet basis $T_A$

**Source:** group theory (traceless symmetric 3×3 matrices, orthonormal)  
**Section:** §4.1

The five basis matrices $T_1,\ldots,T_5$ span $\mathrm{Sym}_0^2(\mathbb{R}^3)$.
At the isotropic point Schur's lemma forces $H_{AB}=2\lambda\,\delta_{AB}$
with $\lambda = 24\pi^2$ (i.e. $\mu_q^2=48\pi^2$).

In [ ]:

# ── Fig. 5 — T_A basis heatmaps (1+2+2 layout) ──────────────────────────
plt.close('all')

vmax = 1.0 / sqrt2 + 0.01
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
axis_labels = ['0', '1', '2']

# Order: m=0 (top, full-width), m=+2, m=-2 (middle row), m=+1, m=-1 (bottom row)
layout_order = [1, 2, 3, 4, 5]
layout_labels = {
    1: r'$T_1\;(m=0)$',
    2: r'$T_2\;(m=+2)$',
    3: r'$T_3\;(m=-2)$',
    4: r'$T_4\;(m=+1)$',
    5: r'$T_5\;(m=-1)$',
}

fig = plt.figure(figsize=(9, 10))
gs  = gridspec.GridSpec(
    3, 2,
    height_ratios=[1.1, 1, 1],
    hspace=0.38, wspace=0.25,
    left=0.07, right=0.88, top=0.91, bottom=0.05,
)

# Shared colorbar axis (right edge)
cbar_ax = fig.add_axes([0.91, 0.08, 0.025, 0.76])

im_last = None
axes_list = [
    fig.add_subplot(gs[0, :]),    # T1 — full-width top
    fig.add_subplot(gs[1, 0]),    # T2
    fig.add_subplot(gs[1, 1]),    # T3
    fig.add_subplot(gs[2, 0]),    # T4
    fig.add_subplot(gs[2, 1]),    # T5
]

for ax, A in zip(axes_list, layout_order):
    mat = T[A]
    im  = ax.imshow(mat, cmap='RdBu_r', norm=norm, aspect='equal')
    im_last = im
    ax.set_title(layout_labels[A], fontsize=11, pad=5)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(axis_labels, fontsize=9)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(axis_labels, fontsize=9)
    for i in range(3):
        for j in range(3):
            val = mat[i, j]
            if abs(val) > 1e-10:
                txt = f'{val:.3f}'.rstrip('0').rstrip('.')
                c   = 'white' if abs(val) > 0.45 else 'black'
                ax.text(j, i, txt, ha='center', va='center',
                        fontsize=8.5, color=c, fontweight='bold')

cbar = fig.colorbar(im_last, cax=cbar_ax)
cbar.set_label('matrix element', fontsize=10)

fig.suptitle(
    r'Fig. 5 — Spin-2 quintet basis $T_A$ (traceless symmetric, $\mathrm{Tr}(T_A T_B)=\delta_{AB}$)',
    fontsize=12, y=0.975,
)

plt.savefig(os.path.join(latex_fig_dir, 'fig05_TA_basis.png'), bbox_inches='tight')
plt.show()
print("Fig. 5 saved.")


## Figs 7, 8, 9 — §4: Spin-2 sector

**Figs 7, 8** use closed-form analytical Hessians (no engine needed):
- $H_{11}^{(q)}, H_{22}^{(q)}$: derived in `spin2_diagonal_hessians.py`
- $H_{44}^{(q)}$: derived in `spin2_offdiag_hessian.py`
- $\varepsilon_{c+} = \sqrt{11/5}-1$ exact (r-independent algebraic condition)

**Fig. 9** requires the FA engine for the order-parameter scan $s_0(\varepsilon)$: `[SLOW]` ~2–4 min first run.

The `[SLOW]` engine cell below is needed **only for Fig. 9**.
Skip to `CFscan` if Fig. 9 is not needed.

**Cache keys:** `spin2_cf_scan` (Figs 7, 8) · `spin2_scan` (Fig. 9)

In [ ]:

# ── FA engine + spin-2 scan (cached) ─────────────────────────
# [SLOW] First run ~2-4 min.  Subsequent runs: instant (cache).

if 'spin2_scan' in cache and 'dist_highres' in cache['spin2_scan']:
    print("Loading spin-2 scan from cache...")
    sc         = cache['spin2_scan']
    r0         = sc['r0']
    eps_pos    = sc['eps_pos']
    H11q_pos   = sc['H11q_pos']
    H22q_pos   = sc['H22q_pos']
    H44q_pos   = sc['H44q_pos']
    eps_c_pos  = sc.get('eps_c_pos', 0.48324)   # brentq-precise value
    eps_neg    = sc['eps_neg']
    a_neg      = sc['a_neg']
    eps_c_neg  = sc.get('eps_c_neg', -0.29309)
    dist_highres     = sc['dist_highres']
    eps_cond_highres = sc['eps_cond_highres']
    s0_highres       = sc['s0_highres']
    print(f"  r0 = {r0:.6f},  eps_c+ = {eps_c_pos:.5f},  eps_c- = {eps_c_neg:.5f}")
    print(f"  Positive: {len(eps_pos)} pts,  Negative: {len(eps_neg)} pts")
    print(f"  High-res s0: {len(s0_highres)} pts")
else:
    if 'spin2_scan' in cache:
        print("Cache found but missing high-res s0 data — re-running engine...")
    else:
        print("[SLOW] Running FA engine + scan (first time only)...")
    t0_total = time.time()

    from sympy import S, cancel, lambdify
    from dppu.topology import CurvatureSource, FiberMode, TopologyType, UnifiedEngine, make_config
    from dppu.torsion.mode import Mode
    from dppu.torsion.nieh_yan import NyVariant
    from dppu.utils.logger import NullLogger

    # ── Engine ──
    t1 = time.time()
    cfg = make_config(
        TopologyType.S3,
        enable_squash=True,
        enable_shear=True,
        fiber_mode=FiberMode.TWIST,
        isotropic_twist=False,
        torsion_mode=Mode.MX,
        ny_variant=NyVariant.FULL,
        weyl_source=CurvatureSource.LC,
        riemann_source=CurvatureSource.LC,
    )
    eng = UnifiedEngine(cfg, NullLogger())
    eng.run()
    print(f"  Engine: {time.time()-t1:.0f}s")

    params   = eng.data['params']
    Veff_sym = eng.data['potential']
    r_s, eps_s, s_s       = params['r'], params['epsilon'], params['s']
    om1_s, om2_s, om3_s   = params['omega1'], params['omega2'], params['omega3']
    V_s, eta_s, L_s       = params['V'], params['eta'], params['L']
    kap_s, th_s, al_s     = params['kappa'], params['theta_NY'], params['alpha']

    # ── Substitute omega=0, physical params ──
    t2 = time.time()
    Veff_w0 = cancel(Veff_sym.subs(
        [(om1_s, S.Zero), (om2_s, S.Zero), (om3_s, S.Zero)]))
    sub_phys = [(V_s, S.One), (eta_s, S.One), (L_s, S.One),
                (kap_s, S.One), (th_s, S.Zero), (al_s, S.Zero)]
    Veff_p = cancel(Veff_w0.subs(sub_phys))
    Veff_num = lambdify((r_s, eps_s, s_s), Veff_p, modules='numpy')
    print(f"  Substitution + lambdify: {time.time()-t2:.0f}s")

    # ── Equilibrium radius ──
    res = minimize_scalar(lambda r: float(Veff_num(r, 0., 0.)),
                          bounds=(0.5, 5.0), method='bounded')
    r0 = res.x
    print(f"  r0 = {r0:.6f}  (expected 3.0)")

    # ── Helper: equilibrium r(eps, s) ──
    def r_eq(eps_v, s_v=0.0):
        return minimize_scalar(
            lambda r: float(Veff_num(r, eps_v, s_v)),
            bounds=(0.3, 6.0), method='bounded', options={'xatol': 1e-8}).x

    # ── H11^q, H22^q (FA engine, with torsion) ──
    def H11q_fn(eps_v, h=1e-4):
        rv = r_eq(eps_v)
        vp = float(Veff_num(rv, eps_v+h, 0.))
        vm = float(Veff_num(rv, eps_v-h, 0.))
        v0 = float(Veff_num(rv, eps_v,   0.))
        return (vp+vm-2*v0)/h**2 * (np.sqrt(6)/2*(1+eps_v))**2

    def H22q_fn(eps_v, h=1e-4):
        rv = r_eq(eps_v)
        vp = float(Veff_num(rv, eps_v, +h))
        vm = float(Veff_num(rv, eps_v, -h))
        v0 = float(Veff_num(rv, eps_v,  0.))
        return (vp+vm-2*v0)/h**2 * 0.5

    # ── H44^q (LC only, numerical Ricci) ──
    def _C_frame(M3, rv):
        eps_t = np.zeros((3,3,3))
        eps_t[0,1,2]=eps_t[1,2,0]=eps_t[2,0,1]=1.
        eps_t[0,2,1]=eps_t[2,1,0]=eps_t[1,0,2]=-1.
        Minv = np.linalg.inv(M3)
        return (4./rv)*np.einsum('ad,be,cf,def->abc', M3, Minv, Minv, eps_t)

    def _R3(C):
        om = 0.5*(np.einsum('abc->cab',C) - C + np.einsum('abc->bac',C))
        R = (np.einsum('cae,edb->cdab',om,om) - np.einsum('cbe,eda->cdab',om,om))
        return np.trace(np.einsum('cdcb->db', R))

    def _Vlc(M3, rv):
        C = _C_frame(M3, rv)
        return -(1./2.) * 2*np.pi**2 * rv**3 * _R3(C)

    def H44q_LC_fn(eps_v, h=1e-3):
        rv = r_eq(eps_v)
        m0 = (1+eps_v)**(1./3.); m2 = (1+eps_v)**(-2./3.)
        M0 = np.diag([m0, m0, m2])
        def V4(q4):
            M = M0 @ (np.eye(3)+q4*T[4]); M /= np.linalg.det(M)**(1./3.)
            return _Vlc(M, rv)
        return (V4(h)+V4(-h)-2*V4(0.))/h**2

    # ── a(eps) = d²V/ds²|_{s=0} ──
    def a_fn(eps_v, h=1e-4):
        rv = r_eq(eps_v)
        vp = float(Veff_num(rv, eps_v, +h))
        vm = float(Veff_num(rv, eps_v, -h))
        v0 = float(Veff_num(rv, eps_v,  0.))
        return (vp+vm-2*v0)/h**2

    # ── Positive eps scan ──
    print("  Positive eps scan (30 pts) ...", end='', flush=True)
    eps_pos  = np.linspace(0.0, 0.55, 30)
    H11q_pos = np.array([H11q_fn(e) for e in eps_pos])
    H22q_pos = np.array([H22q_fn(e) for e in eps_pos])
    H44q_pos = np.array([H44q_LC_fn(e) for e in eps_pos])
    print(" done")

    # ── eps_c+ (brentq for precise value) ──
    sc_idx = np.where(H11q_pos < 0)[0]
    if len(sc_idx):
        i = sc_idx[0]
        eps_c_pos = float(brentq(H11q_fn, eps_pos[i-1], eps_pos[i]))
    else:
        eps_c_pos = 0.48324   # fallback: known from prior analysis
    print(f"  eps_c+ = {eps_c_pos:.5f}  (expected 0.48324)")

    # ── Negative eps scan ──
    print("  Negative eps scan (35 pts) ...", end='', flush=True)
    eps_neg = np.linspace(-0.50, 0.05, 35)
    a_neg   = np.array([a_fn(e) for e in eps_neg])
    print(" done")

    # Find eps_c- precisely
    sc_idx2 = np.where(np.diff(np.sign(a_neg)))[0]
    if len(sc_idx2):
        eps_c_neg = float(brentq(a_fn, eps_neg[sc_idx2[0]], eps_neg[sc_idx2[0]+1]))
    else:
        eps_c_neg = -0.29309
    print(f"  eps_c- = {eps_c_neg:.5f}")

    # ── High-resolution s0 scan in condensed phase ──
    print("  High-res order parameter s0 scan ...", end='', flush=True)

    # Sample 20 log-spaced distances from the critical point in the range 10^-3 to 10^-1
    dist_highres     = np.logspace(-3, -1, 20)
    eps_cond_highres = eps_c_neg - dist_highres

    def s0_fn_highres(eps_v):
        """Dedicated s0 search function with maximally tightened tolerances."""
        def r_eq_strict(e, s):
            return minimize_scalar(
                lambda r: float(Veff_num(r, e, s)),
                bounds=(0.3, 6.0), method='bounded', options={'xatol': 1e-9}
            ).x

        s_g = np.linspace(-0.3, 0.3, 51)
        Vg  = np.array([float(Veff_num(r_eq_strict(eps_v, s), eps_v, s)) for s in s_g])
        sb  = s_g[np.argmin(Vg)]

        res = minimize_scalar(
            lambda s: float(Veff_num(r_eq_strict(eps_v, s), eps_v, s)),
            bounds=(sb-0.1, sb+0.1), method='bounded', options={'xatol': 1e-9}
        )
        return abs(res.x)

    s0_highres = np.array([s0_fn_highres(e) for e in eps_cond_highres])
    print(" done")

    print(f"  Total: {time.time()-t0_total:.0f}s")

    # ── Save cache ──
    cache['spin2_scan'] = dict(
        r0=r0,
        eps_pos=eps_pos, H11q_pos=H11q_pos, H22q_pos=H22q_pos, H44q_pos=H44q_pos,
        eps_c_pos=eps_c_pos,
        eps_neg=eps_neg, a_neg=a_neg, eps_c_neg=eps_c_neg,
        # High-resolution s0 data
        eps_cond_highres=eps_cond_highres, s0_highres=s0_highres, dist_highres=dist_highres,
    )
    save_cache(cache)
    print("Cache saved.")


## [CFscan] Closed-form spin-2 Hessians — for Figs 7, 8

Computes the three representative branches of the spin-2 quintet using closed-form Hessians.
FA engine not required; uses a standalone EC+MX pipeline (~1 s SymPy build + numpy scan).

**Formulas (see App. D.4):**
$$H_{11}^{(q)} = \frac{-160\pi^2 a^6 r + 352\pi^2 r}{3a^8}, \quad
H_{22}^{(q)} = \frac{128\pi^2 a^6 r - 64\pi^2 r}{a^2}, \quad
H_{44}^{(q)} = \frac{32\pi^2 f^{12}r - 48\pi^2 f^6 r + 32\pi^2 f^3 r + 48\pi^2 r}{f^8}$$

$a=(1+\varepsilon)^{1/3}$, $f=a^2$; H₄₄ updated to full-torsion version (previously: LC only).

**Critical points (exact algebraic, $r_0$-independent):**
$\varepsilon_{c+}=\sqrt{11/5}-1$ (H₁₁=0), $\quad\varepsilon_{c-}=1/\sqrt{2}-1$ (H₂₂=0)


In [ ]:
# ── [CFscan] Closed-form spin-2 Hessians — for Figs 7, 8 ─────────────────
# H11/H22 from spin2_diagonal_hessians.py, H44 from spin2_offdiag_hessian.py.
# r_eq(ε) from analytical V_eff(a, r) via standalone EC+MX pipeline (~1 s build).
# ~5 s first run; instant from cache thereafter.

if 'spin2_cf_scan' in cache:
    print("Loading closed-form spin-2 scan from cache...")
    cf      = cache['spin2_cf_scan']
    eps_cf  = cf['eps_cf']
    H11_arr = cf['H11_arr']
    H22_arr = cf['H22_arr']
    H44_arr = cf['H44_arr']
    r0_cf   = cf['r0_cf']
    print(f"  r₀ = {r0_cf:.6f},  {len(eps_cf)} scan points")
else:
    print("Building closed-form scan (first time only, ~5 s)...")
    from sympy import symbols, Symbol, S, Matrix, cancel, pi, lambdify

    # ── Standalone EC+MX pipeline (η=L=κ=1, θ_NY=α=0) ───────────────────
    def _eps3(i, j, k):
        t = (i, j, k)
        if t in {(0,1,2),(1,2,0),(2,0,1)}: return S.One
        if t in {(0,2,1),(2,1,0),(1,0,2)}: return S.NegativeOne
        return S.Zero

    def _build_C(F, Finv, r_s):
        C = [[[S.Zero]*3 for _ in range(3)] for _ in range(3)]
        for i in range(3):
            for j in range(3):
                for k in range(3):
                    v = S.Zero
                    for l in range(3):
                        for m_ in range(3):
                            for n in range(3):
                                e = _eps3(l, m_, n)
                                if e != S.Zero:
                                    v += F[i,l] * Finv[j,m_] * Finv[k,n] * e
                    C[i][j][k] = cancel(4 * v / r_s)
        return C

    def _build_veff(F, Finv, r_s):
        """V_eff = R/(2κ²)·vol via EC+MX pipeline (η=L=κ=1, θ_NY=α=0)."""
        C3 = _build_C(F, Finv, r_s)
        def Cv(a,b,c): return C3[a][b][c] if (a<3 and b<3 and c<3) else S.Zero
        def Ga(a,b,c): return cancel((Cv(a,b,c) + Cv(c,b,a) - Cv(b,a,c)) / 2)
        def Tv(a,b,c):
            t1 = (2/r_s * _eps3(a,b,c) if (a<3 and b<3 and c<3) else S.Zero)
            Vb = S.One if b==3 else S.Zero
            Vc = S.One if c==3 else S.Zero
            t2 = ((S.One if a==c else S.Zero)*Vb - (S.One if a==b else S.Zero)*Vc) / 3
            return t1 + t2
        def Kv(a,b,c): return cancel((Tv(a,b,c) + Tv(b,c,a) - Tv(c,a,b)) / 2)
        GEC = [[[cancel(Ga(a,b,c) + Kv(a,b,c)) for c in range(4)]
                for b in range(4)] for a in range(4)]
        C4  = [[[Cv(a,b,c) for c in range(4)] for b in range(4)] for a in range(4)]
        Rsc = S.Zero
        for a in range(4):
            for b in range(4):
                v = S.Zero
                for e in range(4):
                    v += (GEC[a][e][a] * GEC[e][b][b]
                          - GEC[a][e][b] * GEC[e][b][a]
                          + GEC[a][b][e] * C4[e][a][b])
                Rsc = cancel(Rsc + v)
        return cancel(Rsc / 2 * (2 * pi**2 * r_s**3))

    r_s = symbols('r', positive=True)
    a_s = Symbol('a', positive=True)

    t0 = time.time()
    print("  Building V_eff(a, r) ...", end='', flush=True)
    Veff_sym = _build_veff(
        Matrix.diag(a_s, a_s, 1/a_s**2),
        Matrix.diag(1/a_s, 1/a_s, a_s**2),
        r_s
    )
    Veff_fn = lambdify((a_s, r_s), Veff_sym, modules='numpy')
    print(f" done ({time.time()-t0:.1f}s)")

    def r_eq_cf(eps_v):
        av = (1 + eps_v) ** (1/3)
        return minimize_scalar(
            lambda r: float(Veff_fn(av, r)), bounds=(0.3, 6.), method='bounded').x

    r0_cf = r_eq_cf(0.0)
    print(f"  r₀ = {r0_cf:.6f}  (expected 3.0)")

    # ── Closed-form Hessians (pure numpy) ────────────────────────────────
    def H11_cf(eps_v, r_v):
        a = (1 + eps_v) ** (1/3)
        return (-160*np.pi**2*a**6*r_v + 352*np.pi**2*r_v) / (3*a**8)

    def H22_cf(eps_v, r_v):
        a = (1 + eps_v) ** (1/3)
        return (128*np.pi**2*a**6*r_v - 64*np.pi**2*r_v) / a**2

    def H44_cf(eps_v, r_v):
        f = (1 + eps_v) ** (2/3)
        return (32*np.pi**2*f**12*r_v - 48*np.pi**2*f**6*r_v
                + 32*np.pi**2*f**3*r_v + 48*np.pi**2*r_v) / f**8

    print("  Scanning 60 pts ...", end='', flush=True)
    eps_cf  = np.linspace(0.0, 0.55, 60)
    H11_arr = np.zeros(60)
    H22_arr = np.zeros(60)
    H44_arr = np.zeros(60)
    for i, e in enumerate(eps_cf):
        rv = r_eq_cf(e)
        H11_arr[i] = H11_cf(e, rv)
        H22_arr[i] = H22_cf(e, rv)
        H44_arr[i] = H44_cf(e, rv)
        print('.', end='', flush=True)
    print(" done")

    cache['spin2_cf_scan'] = dict(
        eps_cf=eps_cf, H11_arr=H11_arr, H22_arr=H22_arr, H44_arr=H44_arr, r0_cf=r0_cf)
    save_cache(cache)
    print("Cache saved.")

# ── Derived constants ─────────────────────────────────────────────────────
H_schur_cf      = 64 * np.pi**2 * r0_cf
eps_c_pos_exact = np.sqrt(11./5.) - 1.    # H11=0: 5a⁶=11 (exact, r-independent)
eps_c_neg_exact = 1./np.sqrt(2.) - 1.     # H22=0: 2a⁶=1  (exact, r-independent)

print(f"  Schur ref  64π²r₀ = {H_schur_cf:.2f}")
print(f"  ε_c+ (exact) = {eps_c_pos_exact:.6f}  [H11=0: 5a⁶=11]")
print(f"  ε_c- (exact) = {eps_c_neg_exact:.6f}  [H22=0: 2a⁶=1]")

## Fig. 7 — §4: Spin-2 mass splitting under squash background

Closed-form Hessian eigenvalues $H_{AA}^{(q)}(\varepsilon)$ for the three SO(2) irreps of the spin-2 quintet:

| mode | basis | closed-form expression |
|---|---|---|
| $m=0$ | $T_1$ singlet | $H_{11}^{(q)} = \dfrac{-160\pi^2 a^6 r + 352\pi^2 r}{3a^8}$ |
| $m=\pm2$ | $T_2,T_3$ doublet | $H_{22}^{(q)} = \dfrac{128\pi^2 a^6 r - 64\pi^2 r}{a^2}$ |
| $m=\pm1$ | $T_4,T_5$ doublet | $H_{44}^{(q)} = \dfrac{32\pi^2 f^{12} r - 48\pi^2 f^{6} r + 32\pi^2 f^{3} r + 48\pi^2 r}{f^{8}}$ |

where $a=(1+\varepsilon)^{1/3}$, $f=a^2=(1+\varepsilon)^{2/3}$; physical params $\eta=L=\kappa=1$, $\alpha=\theta_\text{NY}=0$, Mode.MX.  
All three include full EC+torsion (no approximation).

**Schur degeneracy at $\varepsilon=0$:**
$$H_{11}^{(q)} = H_{22}^{(q)} = H_{44}^{(q)} = 64\pi^2 r_0 \approx 1895 \qquad (r_0=3)$$

**Exact critical point** (algebraic, $r_0$-independent):
$$H_{11}^{(q)}=0 \iff 5a^6=11 \implies \varepsilon_{c+}=\sqrt{\tfrac{11}{5}}-1\approx 0.4833$$

The $m=0$ singlet softens to zero at $\varepsilon_{c+}$; the $m=\pm2$ and $m=\pm1$ doublets remain massive throughout.

In [ ]:
# ── Fig. 7 — spin-2 mass splitting (closed-form analytical) ──────────────
plt.close('all')

fig, ax = plt.subplots(figsize=(7.5, 5))

ax.plot(eps_cf, H11_arr, color='tab:blue',   lw=2.2,
        label=r'$H_{11}^{(q)}$  $(m=0,\; T_1)$')
ax.plot(eps_cf, H22_arr, color='tab:orange', lw=2.2, ls='--',
        label=r'$H_{22}^{(q)}$  $(m=\pm2,\; T_2/T_3)$')
ax.plot(eps_cf, H44_arr, color='tab:green',  lw=2.0, ls='-.',
        label=r'$H_{44}^{(q)}$  $(m=\pm1,\; T_4/T_5)$')

# Schur degeneracy line
ax.axhline(H_schur_cf, color='gray', lw=0.9, ls=':', alpha=0.7)
ax.text(0.01, H_schur_cf + 20,
        rf'$H_{{AA}}^{{(q)}}(0)=64\pi^2 r_0\approx{H_schur_cf:.0f}$ (Schur)',
        fontsize=10, color='gray', va='bottom')

# Exact eps_c+ vertical line
ax.axvline(eps_c_pos_exact, color='tab:blue', lw=1.2, ls=':', alpha=0.6)
ax.text(eps_c_pos_exact + 0.005, 200,
        r'$\varepsilon_{c+}=\sqrt{11/5}-1$' + f'\n$\\approx{eps_c_pos_exact:.4f}$',
        fontsize=10, color='tab:blue', rotation=90, va='bottom')

ax.axhline(0, color='k', lw=0.7)
ax.set_xlabel(r'$\varepsilon$  (squash parameter)')
ax.set_ylabel(r'$H_{AA}^{(q)}(\varepsilon)$')
ax.set_title(r'Fig. 7 — Spin-2 mass splitting under squash background $\varepsilon$')
ax.legend(loc='upper right')
ax.set_xlim(eps_cf[0], eps_cf[-1])

plt.tight_layout()
plt.savefig(os.path.join(latex_fig_dir, 'fig07_spin2_splitting.png'), bbox_inches='tight')
plt.show()
print("Fig. 7 saved.")
print(f"  r₀ = {r0_cf:.6f},  ε_c+ (exact) = {eps_c_pos_exact:.6f}")
print(f"  H11(0) = {H11_arr[0]:.2f},  H22(0) = {H22_arr[0]:.2f},  H44(0) = {H44_arr[0]:.2f}")

## Fig. 8 — §4: $T_1$ soft mode at $\varepsilon_{c+}$

Left: closed-form $H_{11}^{(q)}(\varepsilon)$ approaching zero at the exact critical point
$$\varepsilon_{c+} = \sqrt{\tfrac{11}{5}}-1 \approx 0.4833 \quad\text{(exact algebraic condition } 5a^6=11\text{, }r_0\text{-independent)}$$

Right: ratio $H_{22}^{(q)} / H_{11}^{(q)}$ diverging as $\varepsilon\to\varepsilon_{c+}$,
confirming that the $m=\pm2$ doublet remains massive throughout the phase transition.

In [ ]:
# ── Fig. 8 — T1 soft mode + H22/H11 ratio (closed-form) ─────────────────
plt.close('all')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel (a): H11^q vs eps
ax1.plot(eps_cf, H11_arr, color='tab:blue', lw=2.2)
ax1.axhline(0, color='k', lw=0.7)
ax1.axvline(eps_c_pos_exact, color='tab:blue', lw=1.2, ls='--', alpha=0.7)
ax1.text(eps_c_pos_exact + 0.005, max(H11_arr) * 0.05,
         r'$\varepsilon_{c+}=\sqrt{11/5}-1$' + f'\n$\\approx {eps_c_pos_exact:.4f}$',
         fontsize=10, color='tab:blue', rotation=90, va='bottom')
ax1.set_xlabel(r'$\varepsilon$')
ax1.set_ylabel(r'$H_{11}^{(q)}(\varepsilon)$')
ax1.set_title(r'(a) $T_1$ soft mode ($m=0$ singlet)')
ax1.set_xlim(eps_cf[0], eps_cf[-1])

# Panel (b): ratio H22/H11
ratio = np.where(np.abs(H11_arr) > 10, H22_arr / H11_arr, np.nan)
ax2.plot(eps_cf, ratio, color='tab:orange', lw=2.2)
ax2.axvline(eps_c_pos_exact, color='tab:blue', lw=1.2, ls='--', alpha=0.7)
ax2.axhline(1.0, color='gray', lw=0.9, ls=':', alpha=0.6)
ax2.text(0.01, 1.05, r'ratio $=1$ at $\varepsilon=0$ (Schur)', fontsize=9, color='gray')
ax2.set_xlabel(r'$\varepsilon$')
ax2.set_ylabel(r'$H_{22}^{(q)} / H_{11}^{(q)}$')
ax2.set_title(r'(b) $m=\pm2$ doublet remains massive')
ax2.set_xlim(eps_cf[0], eps_cf[-1])
ax2.set_ylim(0, 20)

fig.suptitle(
    r'Fig. 8 — $T_1$ soft mode identification at $\varepsilon_{c+}=\sqrt{11/5}-1\approx0.4833$',
    fontsize=13
)
plt.tight_layout()
plt.savefig(os.path.join(latex_fig_dir, 'fig08_soft_mode.png'), bbox_inches='tight')
plt.show()
print("Fig. 8 saved.")

## Fig. 9 — §4: $s\neq0$ phase transition (negative-$\varepsilon$ side)

Left: $a(\varepsilon)=\partial^2 V/\partial s^2\big|_{s=0}$ vs $\varepsilon$,
crossing zero at $\varepsilon_{c-}\approx -0.293$.

Right: order parameter $s_0(\varepsilon)=\arg\min_s F(s;\varepsilon)$
vs $|\varepsilon-\varepsilon_{c-}|$ on a log–log scale with the
mean-field fit $s_0\propto|\varepsilon-\varepsilon_{c-}|^\beta$,
$\beta\approx0.47$ ($R^2\approx0.987$).
Data sampled at 20 log-spaced distances from the critical point ($10^{-3}$–$10^{-1}$).


In [ ]:

# ── Fig. 9 — s≠0 phase transition ────────────────────────────
plt.close('all')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Load high-res variables from cache (robust for standalone execution) ──
sc = cache['spin2_scan']
dist_highres     = sc['dist_highres']
eps_cond_highres = sc['eps_cond_highres']
s0_highres       = sc['s0_highres']

# ── Panel (a): a(eps) = d²V/ds²|_{s=0} ──
ax1.plot(eps_neg, a_neg, color='tab:purple', lw=2.2)
ax1.axhline(0, color='k', lw=0.8)
ax1.axvline(eps_c_neg, color='tab:purple', lw=1.3, ls='--', alpha=0.7)
ax1.text(eps_c_neg + 0.008, max(a_neg)*0.08,
         rf'$\varepsilon_{{c-}} \approx {eps_c_neg:.4f}$',
         fontsize=10, color='tab:purple', rotation=90, va='bottom')

# Fill unstable region (a < 0)
eps_unstable = eps_neg[a_neg < 0]
a_unstable   = a_neg[a_neg < 0]
if len(eps_unstable):
    ax1.fill_betweenx([min(a_neg)*1.05, 0],
                      eps_unstable[0], eps_unstable[-1],
                      alpha=0.12, color='tab:purple', label=r'$s\neq0$ phase')
ax1.set_xlabel(r'$\varepsilon$')
ax1.set_ylabel(r'$a(\varepsilon) = \partial^2 V/\partial s^2|_{s=0}$')
ax1.set_title(r'(a) Instability onset at $\varepsilon_{c-}$')
ax1.legend(loc='upper right', fontsize=10)
ax1.set_xlim(eps_neg[0], eps_neg[-1]+0.01)

# ── Panel (b): order parameter s0 vs |eps - eps_c-| (high-res) ──
# Exclude very near-critical points (<10^-3) and unconverged s0 values from the fit
valid_mask  = (dist_highres > 2e-3) & (s0_highres > 1e-5)
d_fit_data  = dist_highres[valid_mask]
s0_fit_data = s0_highres[valid_mask]

if len(s0_fit_data) >= 4:
    log_d = np.log(d_fit_data)
    log_s = np.log(s0_fit_data)
    # Linear regression: log(s0) = beta log(|eps-eps_c|) + const
    coeffs   = np.polyfit(log_d, log_s, 1)
    beta_fit = coeffs[0]
    A_fit    = np.exp(coeffs[1])
    R2 = 1 - np.sum((log_s - np.polyval(coeffs, log_d))**2) / np.var(log_s) / len(log_s)

    # Plot all high-resolution data points
    ax2.loglog(dist_highres, s0_highres, 'o', color='tab:purple', ms=6, label='data')

    # Fit line
    d_line = np.logspace(np.log10(dist_highres.min()), np.log10(dist_highres.max()), 100)
    ax2.loglog(d_line, A_fit * d_line**beta_fit, 'k--', lw=1.8,
               label=rf'$s_0 \propto |\varepsilon-\varepsilon_{{c-}}|^{{\beta}}$'
                     f'\n' + rf'$\beta={beta_fit:.4f},\; R^2={R2:.3f}$')
    ax2.legend(fontsize=10)
    print(f"  beta = {beta_fit:.4f},  A = {A_fit:.4f}, R2 = {R2:.4f}")
else:
    ax2.text(0.5, 0.5, 'Insufficient condensed-phase data\n(re-run engine cell)',
             transform=ax2.transAxes, ha='center', fontsize=10, color='gray')

ax2.set_xlabel(r'$|\varepsilon - \varepsilon_{c-}|$')
ax2.set_ylabel(r'order parameter $s_0$')
ax2.set_title(r'(b) Order parameter $s_0 \propto |\varepsilon-\varepsilon_{c-}|^\beta$')

fig.suptitle(
    r'Fig. 9 — $s\neq0$ phase transition on the negative-$\varepsilon$ side'
    r'  ($\varepsilon_{c-} \approx -0.293$)',
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(latex_fig_dir, 'fig09_sneg_phase.png'), bbox_inches='tight')
plt.show()
print("Fig. 9 saved.")


## Figs 11, 12 — §5: Spin-1 sector

### Fig. 11: $G$ field-space metric eigenvalue spectrum
### Fig. 12: $(\omega, \delta) \to (X, Y)$ basis rotation schematic

Fig. 11 requires the velocity-mode engine (`enable_velocity=True`): **[VERY SLOW]** ~300–600 s first run.  
Fig. 12 is a schematic (no engine needed).

In [ ]:
# ── Velocity engine — G metric (cached) ──────────────────────
# [VERY SLOW] First run: ~300–600 s.  Subsequent: instant (cache).

if 'G_metric' in cache:
    print("Loading G metric from cache...")
    G_num = cache['G_metric']
    print(f"  G (6×6) loaded.  Eigenvalues: {np.linalg.eigvalsh(G_num).round(3)}")
else:
    print("[VERY SLOW] Running velocity-mode engine (~300–600 s)...")
    t0_G = time.time()

    from dppu.topology.unified import UnifiedEngine, DOFConfig, TopologyType, FiberMode
    from dppu.torsion.mode import Mode
    from dppu.torsion.nieh_yan import NyVariant
    from sympy import S, diff, lambdify

    cfg = DOFConfig(
        topology=TopologyType.S3,
        enable_squash=False,
        enable_shear=False,
        fiber_mode=FiberMode.BOTH,
        isotropic_twist=False,
        torsion_mode=Mode.MX,
        ny_variant=NyVariant.FULL,
        enable_velocity=True,
    )
    engine = UnifiedEngine(cfg)
    engine.run()
    t1_G = time.time()
    print(f"  Engine: {t1_G-t0_G:.0f}s")

    fp   = engine.get_free_params()
    Veff = engine.data['potential']

    v_syms = [fp.get(k) for k in
              ['v_omega1', 'v_omega2', 'v_omega3', 'v_delta0', 'v_delta1', 'v_delta2']]

    # Physical parameter substitution
    phys = {}
    for key, val in [
        ('eta', 1), ('V', 1), ('L', 1), ('kappa', 1),
        ('alpha', 0), ('theta_NY', 0),
        ('omega1', 0), ('omega2', 0), ('omega3', 0),
        ('delta0', 0), ('delta1', 0), ('delta2', 0),
        ('cd0', 1), ('sd0', 0), ('cd1', 1), ('sd1', 0), ('cd2', 1), ('sd2', 0),
    ]:
        sym = fp.get(key)
        if sym is not None:
            phys[sym] = val

    Viso = Veff.subs(phys)
    v_zero = {v: 0 for v in v_syms}
    r_sym  = fp['r']

    # r0
    Vstatic = lambdify(r_sym, Viso.subs(v_zero), 'numpy')
    r0_G = minimize_scalar(lambda r: float(Vstatic(r)), bounds=(1., 10.), method='bounded').x
    print(f"  r0 = {r0_G:.4f}")

    Vr0 = Viso.subs(r_sym, r0_G)

    # G_{ij} = (1/2) d²V / dv_i dv_j |_{v=0}
    G_sym = [[None]*6 for _ in range(6)]
    for i, vi in enumerate(v_syms):
        for j, vj in enumerate(v_syms):
            if j < i:
                G_sym[i][j] = G_sym[j][i]
            else:
                G_sym[i][j] = diff(diff(Vr0, vi), vj).subs(v_zero) / 2

    G_num = np.array([[float(G_sym[i][j]) for j in range(6)] for i in range(6)])
    print(f"  G computed in {time.time()-t1_G:.0f}s")
    print(f"  Eigenvalues: {np.linalg.eigvalsh(G_num).round(3)}")

    cache['G_metric'] = G_num
    save_cache(cache)
    print("Cache saved.")

## Fig. 11 — §5: $G$ eigenvalue spectrum

The $6\times6$ field-space metric $G$ has rank 3:
three zero eigenvalues (null directions = $Y_k$) and
three degenerate positive eigenvalues $\approx 769.83$ (non-null kinetic coordinate = $X_k$).

In [ ]:
# ── Fig. 11 — G eigenvalue spectrum ──────────────────────────
plt.close('all')
G_eigs = np.linalg.eigvalsh(G_num)

fig, ax = plt.subplots(figsize=(7, 4.5))

colors = ['tab:red'  if abs(e) < 10 else 'tab:blue' for e in G_eigs]
labels = [r'null $\to Y_k$' if abs(e) < 10 else r'non-null kinetic $\to X_k$' for e in G_eigs]

bars = ax.bar(np.arange(6), G_eigs, color=colors, edgecolor='k', linewidth=0.8, width=0.6)

# Legend proxies
from matplotlib.patches import Patch
legend_handles = [
    Patch(color='tab:red',  label=r'null mode ($G$ eigenvalue $=0$, $Y_k$)'),
    Patch(color='tab:blue', label=r'non-null kinetic coordinate ($G$ eigenvalue $>0$, $X_k$)'),
]
ax.legend(handles=legend_handles, fontsize=11)

# Annotate values
for i, e in enumerate(G_eigs):
    txt = f'{e:.1f}'
    ax.text(i, max(e, 5) + 15, txt, ha='center', va='bottom', fontsize=9)

ax.set_xticks(np.arange(6))
ax.set_xticklabels([rf'$g_{i+1}$' for i in range(6)], fontsize=11)
ax.set_ylabel(r'$G$ eigenvalue')
ax.set_ylim(-30, G_eigs.max() * 1.18)

ax.set_title(
    r'Fig. 11 — Eigenvalue spectrum of $G$ (field-space metric)'
    '\n' + r'3 null ($\to Y_k$) + 3 non-null kinetic coordinates ($\to X_k$), rank$\,(G)=3$',
    fontsize=12
)

plt.tight_layout()
plt.savefig(os.path.join(latex_fig_dir, 'fig11_G_spectrum.png'), bbox_inches='tight')
plt.show()
print("Fig. 11 saved.")

## Fig. 12 — §5: $(\omega, \delta) \to (X, Y)$ basis rotation

Schematic of the spin-1 sector in the 2D $(\omega_k, \delta_k)$ plane
(per axis $k$, drawn for $k=0$).

- **$Y$ direction** (null of $G$): $(1,\; 2L/r_0)=(1,\; 2/3)$ — non-dynamical auxiliary  
- **$X$ coordinate**: $(-2,\; 3)/\sqrt{13}$ — non-null kinetic coordinate before auxiliary elimination  
- **Corrected physical mode**: approximately $(0.9816,\;0.1909)$ with $\lambda_\mathrm{phys}\simeq -0.4183<0$ and positive kinetic norm

The rotation is orthogonal in the $G$-metric, not in the Euclidean metric.

In [ ]:
# ── Fig. 12 — (omega, delta) → (X, Y) basis rotation schematic ──────
plt.close('all')
r0_val = 3.0
L_val  = 1.0

# Y direction (null of G): (1, 2L/r0) — normalize for display
Y_raw = np.array([1.0, 2*L_val/r0_val])      # (1, 2/3)
Y_dir = Y_raw / np.linalg.norm(Y_raw)

# X coordinate before auxiliary elimination: (-2, 3)/sqrt(13)
X_raw = np.array([-2.0, 3.0])
X_dir = X_raw / np.linalg.norm(X_raw)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')

# Coordinate axes
for vec, name, col in [(np.array([1,0]), r'$\omega_k$', 'gray'),
                        (np.array([0,1]), r'$\delta_k$', 'gray')]:
    ax.annotate('', xy=1.3*vec, xytext=np.zeros(2),
                arrowprops=dict(arrowstyle='->', color=col, lw=1.4))
    ax.text(*(1.35*vec), name, ha='center', va='center', fontsize=13, color=col)

# Y direction (null)
scale = 1.15
ax.annotate('', xy=scale*Y_dir, xytext=np.zeros(2),
            arrowprops=dict(arrowstyle='->', color='tab:red', lw=2.2))
ax.text(*(scale*Y_dir + np.array([0.06, 0.06])),
        r'$Y_k = (1,\;2/3)/\|.\|$' + '\n' + r'(null, $G\cdot Y=0$)',
        fontsize=10.5, color='tab:red', ha='left', va='bottom')

# Corrected physical spin-1 mode after Schur elimination
X_corr = np.array([0.98161543, 0.19086943])
X_corr = X_corr / np.linalg.norm(X_corr)

# X coordinate before auxiliary elimination
ax.annotate('', xy=scale*X_dir, xytext=np.zeros(2),
            arrowprops=dict(arrowstyle='->', color='tab:blue', lw=1.5, linestyle='--', alpha=0.55))
ax.text(*(scale*X_dir + np.array([-0.07, -0.09])),
        r'$X_k=(-2,3)/\sqrt{13}$' + '\n' + r'(pre-Schur coordinate)',
        fontsize=9.5, color='tab:blue', ha='right', va='top', alpha=0.75)

ax.annotate('', xy=scale*X_corr, xytext=np.zeros(2),
            arrowprops=dict(arrowstyle='->', color='tab:green', lw=2.4))
ax.text(*(scale*X_corr + np.array([-0.02, -0.3])),
        r'$v_\mathrm{corr}\simeq(0.9816,0.1909)$' + '\n' + r'$\lambda_\mathrm{phys}\simeq-0.4183<0$' + '\n' + r'(tachyonic, non-ghost)',
        fontsize=10.0, color='tab:green', ha='right', va='top')

# Origin marker
ax.plot(0, 0, 'ko', ms=5)

# Right-angle indicator between X and Y
angle_scale = 0.12
# Draw small arc to show they are G-orthogonal (not Euclidean)
theta_Y = np.arctan2(Y_dir[1], Y_dir[0])
theta_X = np.arctan2(X_dir[1], X_dir[0])
thetas  = np.linspace(theta_Y, theta_X, 40)
ax.plot(angle_scale*np.cos(thetas), angle_scale*np.sin(thetas),
        color='gray', lw=1.0, ls='--')
ax.text(angle_scale*0.8*np.cos((theta_Y+theta_X)/2),
        angle_scale*0.8*np.sin((theta_Y+theta_X)/2),
        r'$G$-orth.', fontsize=8.5, color='black', ha='center')

ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-1.6, 1.6)
ax.set_title(
    r'Fig. 12 — Spin-1 basis rotation: $(\omega_k,\delta_k)\to(X_k, Y_k)$'
    '\n' + r'(single-axis slice, $k=0$;  $r_0=3$, $L=1$)',
    fontsize=12
)
ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(latex_fig_dir, 'fig12_spin1_rotation.png'), bbox_inches='tight')
plt.show()
print("Fig. 12 saved.")

## Summary

All figures generated:

| Figure | File | Description |
|--------|------|-------------|
| Fig. 4 | `fig04_pontryagin_mx.png` | MX Pontryagin density $P_0(r)$ |
| Fig. 5 | `fig05_TA_basis.png` | Spin-2 $T_A$ basis heatmaps |
| Fig. 7 | `fig07_spin2_splitting.png` | Mass splitting vs $\varepsilon$ |
| Fig. 8 | `fig08_soft_mode.png` | $T_1$ soft mode at $\varepsilon_{c+}$ |
| Fig. 9 | `fig09_sneg_phase.png` | $s\neq0$ phase transition |
| Fig. 11 | `fig11_G_spectrum.png` | $G$ eigenvalue spectrum |
| Fig. 12 | `fig12_spin1_rotation.png` | $(\omega,\delta)\to(X,Y)$ schematic |

All output saved to `../LaTeX/figures/` for the paper draft.